# Çürük modelini Colab'da eğit

Bu defter Google'ın ücretsiz GPU'sunda çürük tespit modelini eğitir ve sonunda `best.pt` dosyasını bilgisayarına indirir.

**Başlamadan önce:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç ve kaydet.

Sonra **Çalışma zamanı → Tümünü çalıştır** de. Toplam süre yaklaşık 1-2 saat. Sekmeyi kapatma; uzun süre dokunulmazsa Colab oturumu kapatabilir.

> Araştırma prototipidir, klinik karar için değildir.

## 1. GPU var mı?

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç, sonra yeniden çalıştır."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Kodu ve kütüphaneleri indir

In [ ]:
import os, subprocess
REPO = "https://github.com/sahinemin751-cmyk/Tasarla-di-hekimi-yapay-zeka-asistan.git"
if not os.path.exists("/content/asistan"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/asistan"], check=True)
%cd /content/asistan
# Her çalıştırmada kodun en güncel hâlini al (PR birleştirilmediyse prototip dalı)
!git fetch -q --depth 1 origin +main:refs/remotes/origin/main
!git fetch -q --depth 1 origin +caries-prototip:refs/remotes/origin/caries-prototip 2>/dev/null || true
!git checkout -q origin/caries-prototip 2>/dev/null || git checkout -q origin/main
!git log --oneline -1
!pip install -q ultralytics

## 3. Veri setini indir ve hazırla (Zenodo "Benchmarking Dataset", 1,6 GB, ~10 dakika)

Veri setinin yazarlarının hazırladığı train / valid / test ayrımlı sürümü kullanıyoruz; sonuçlarımız makaleyle karşılaştırılabilir olur.

In [ ]:
ZIP = "/content/Benchmarking Dataset.zip"
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/14827784/files/Benchmarking%20Dataset.zip?download=1"
!rm -rf /content/data/caries
!python scripts/prepare_caries_dataset.py --source "{ZIP}" --out /content/data/caries

Yukarıdaki çıktıda **Etiket kaynağı: COCO** yazmalı ve sınıf adlarıyla kutu sayıları listelenmeli. Öyle değilse bu çıktının ekran görüntüsünü Claude'a gönder.

## 4. Modeli eğit (en uzun adım, ~1-2 saat)

Daha hızlı denemek için `EPOCHS` değerini 20 yapabilirsin; sonuç biraz daha zayıf olur.

In [ ]:
EPOCHS = 60
!python scripts/train_caries.py --data /content/data/caries/data.yaml --epochs {EPOCHS} --device 0
if _exit_code != 0:
    raise RuntimeError("Eğitim hata verdi. Bu kutunun çıktısının SONUNU (kırmızı satırlar) Claude'a gönder.")


## 5. Birkaç test görüntüsünde sonucu gör

In [ ]:
import glob, os
from ultralytics import YOLO
from IPython.display import Image, display
found = ["runs/caries/weights/best.pt"] if os.path.exists("runs/caries/weights/best.pt") else sorted(glob.glob("runs/**/best.pt", recursive=True), key=os.path.getmtime)
if not found:
    raise RuntimeError("best.pt yok: 4. adımdaki eğitim tamamlanmamış. 4. adımın çıktısının sonunu Claude'a gönder.")
best = found[-1]
print("Model:", best)
model = YOLO(best)
for path in sorted(glob.glob("/content/data/caries/images/test/*"))[:6]:
    out = model.predict(path, conf=0.35, verbose=False)[0]
    out.save(filename="/content/tahmin.jpg")
    display(Image("/content/tahmin.jpg", width=480))

## 6. Modeli bilgisayarına indir

İnen `best.pt` dosyasını proje klasöründe `runs\caries\weights\` içine koy, sonra kamera demosunu kendi bilgisayarında çalıştır:

```
python scripts\camera_demo.py --weights runs\caries\weights\best.pt
```

Ekran kartı olmadan da çalışır; sadece kare hızı daha düşük olur.

In [ ]:
from google.colab import files
files.download(best)